# HybridDataset walkthrough

Minimal, valid notebook showing HybridDataset mixed routing and source overrides.

In [1]:
from __future__ import annotations

import asyncio
import datetime as dt
import os
import threading
from pathlib import Path
from tempfile import TemporaryDirectory

from sqlalchemy import Date, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import DataHelper, HybridDataset, SqlDatabaseConfig


def run_async_from_sync(coro):
    """Run a coroutine from sync code, including inside notebook event loops."""
    result: dict[str, object] = {}
    error: dict[str, BaseException] = {}

    def _runner() -> None:
        try:
            result["value"] = asyncio.run(coro)
        except BaseException as exc:  # Bubble up original exception after thread join.
            error["value"] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()

    if "value" in error:
        raise error["value"]
    return result["value"]


class Base(DeclarativeBase):
    pass


class HistoricalEvent(Base):
    __tablename__ = "historical_events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))


class LiveEvent(Base):
    __tablename__ = "live_events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))


In [2]:
tmp_dir = TemporaryDirectory()
db_path = Path(tmp_dir.name) / "hybrid_notebook.db"
engine = create_engine(f"sqlite:///{db_path}")

# Resolve the worker DSN env var name at runtime so it is not hardcoded per helper.
worker_dsn_env_var = os.getenv("BOTI_WORKER_DSN_ENV_VAR") or f"BOTI_SQL_DSN_{db_path.stem.upper()}"
os.environ[worker_dsn_env_var] = f"sqlite:///{db_path}"

Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([
        HistoricalEvent(id=1, event_date=dt.date(2026, 4, 14), status="hist"),
        HistoricalEvent(id=2, event_date=dt.date(2026, 4, 16), status="hist"),
        LiveEvent(id=10, event_date=dt.date(2026, 4, 18), status="live"),
        LiveEvent(id=11, event_date=dt.date(2026, 4, 19), status="live"),
    ])
    session.commit()


In [3]:
sql_config = SqlDatabaseConfig(
    connection_url=f"sqlite:///{db_path}",
    worker_connection_env_var=worker_dsn_env_var,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
)

historical = DataHelper(sql_config, table="historical_events")
live = DataHelper(sql_config, table="live_events")
dataset = HybridDataset(historical, live, date_field="event_date", split_date="2026-04-18")


In [4]:
mixed = dataset.load(start="2026-04-14", end="2026-04-19", return_type="auto")
historical_df = dataset.pandas.load(start="2026-04-14", end="2026-04-17", source="historical")
live_df = dataset.pandas.load(start="2026-04-18", end="2026-04-19", source="live")
async_df = run_async_from_sync(dataset.aload(start="2026-04-16", end="2026-04-18", return_type="pandas"))

mixed_rows = mixed.shape[0]
if hasattr(mixed_rows, "compute"):
    mixed_rows = mixed_rows.compute()

summary = {
    "mixed_type": type(mixed).__name__,
    "mixed_rows": int(mixed_rows),
    "historical_rows": len(historical_df),
    "live_rows": len(live_df),
    "async_rows": len(async_df),
}
summary


{'mixed_type': 'DataFrame',
 'mixed_rows': 4,
 'historical_rows': 2,
 'live_rows': 2,
 'async_rows': 2}

In [5]:
dataset.close()
engine.dispose()
os.environ.pop(worker_dsn_env_var, None)
tmp_dir.cleanup()
